# 📚 数据分析终极速查小抄本 (Cheat Sheet)

> **使用说明 (写在最前)**：
> 不要死记硬背！这就是你的“外挂”和“库房”。遇到记不清的函数，直接来这里 **Ctrl+C / Ctrl+V**。
>
> **核心心法**：脑子里只有“大白话（动作）” ➡️ 对应找“代码片段”。

---
## 🎯 目录
1. **基础环境与配置**：导库、文件读取、显示配置
2. **快速选数 (定位格)**：`loc`, `iloc` 定位、筛选与修改
3. **数据拼接 (搭积木)**：四种 JOIN 速查、重命名
4. **统计分析 (分类看)**：`GroupBy` 分组、均值/计数
5. **底层运算 (找 NumPy)**：`arange` 造数、`reshape`、超级标签 `np.select`/`np.where`
6. **SQL 高级分析**：窗口函数 (OVER, PARTITION BY, 等)
7. **行列转换 (变宽/变长)**：`pivot_table` 透视表与 `melt` 反向透视
8. **文本处理 (字符串)**：提取、替换、正则表达式
9. **时间序列分析**：转换格式、滚动求和、滞后
10. **数据可视化 (快速画图)**：箱线图、热力图、趋势图
11. **机器学习 (算法模板)**：K-Means 聚类、线性回归
12. **📂 自动化办公 (批量处理)**：文件名遍历、Excel 转 CSV、os 库操作
13. **数据库高级**：游标 (Cursor) 与 CTAS 建表
14. **全量沙盒**：笛卡尔积 (Cross Join)
15. **GroupBy 百分比化**：自定义格式转换、动态列生成
16. **精准行列筛选**：isin 点名式过滤、索引 vs 列的区别
15. **GroupBy 百分比化**：自定义格式转换、动态列生成
16. **精准行列筛选**：isin 点名式过滤、索引 vs 列的区别

---
## 1️⃣ 基础环境与通用预处理

把导入库和第一步的显示配置、最基础的“空值/重复值清洗”存这里。

In [ ]:
import pandas as pd
import numpy as np

# 💡 强迫症必备配置：设置后 Pandas 显示长表时不会省略列名
pd.set_option('display.max_columns', None) 
# pd.set_option('display.max_rows', 100) # (可选) 最多显示行数

# ============================
# 👉 【代码积木】快速重置与去重
# ============================
def clean_basic(df):
    """一个简单的日常‘洗基础数据’小工具"""
    df = df.drop_duplicates()             # 去重
    df = df.dropna(subset=['必须有值的列名']) # 删除特定列为空的行 (注意换列名)
    df = df.reset_index(drop=True)        # 把前面弄乱的 1,2,3,4 序号重新排整齐
    return df

---
## 2️⃣ 快速选数：我到底该怎么查某一行/修改某一个？(`loc` & `iloc`)

> **大白话口诀**：
> *   `loc`：用**名字/标签**找（或者扔个 `True/False` 条件进去找）。
> *   `iloc`：用**数字编号号第几格**找。

In [ ]:
# 假设 df 是我们读进来的表

# 1. 🔍 [查询]：只看 "销量" 前 5 行 (SQL: SELECT 销量 FROM df LIMIT 5)
# display( df[['销量']].head(5) )  

# 2. 🔍 [多条件查询 loc]：找"评价为暴利" 且 "零件为内存条" 的记录
#    注意必须用 (), 且用 '&' (且) 或者是 '|' (或)。
# result = df.loc[ (df['评价'] == '暴利') & (df['硬件类型'] == '内存条') ]

# 3. ✍️ [安全赋值 loc]：当 "内存量" 大于 48 时，给那一行的 "Tier_Level" 列打上 "顶级旗舰" (SQL: CASE WHEN)
# df.loc[df['内存(G)'] > 48, 'Tier_Level'] = '顶级旗舰'
# df.loc[df['内存(G)'] <= 48, 'Tier_Level'] = '常规机型'

# 4. 🔪 [纯数字盲切 iloc]：不管列名叫啥，我就是要前两行、前三列的数据查底看一眼！
# quick_view = df.iloc[0:2, 0:3] 

---
## 3️⃣ 数据拼图：让 A 表认识 B 表 (`pd.merge` 对应 SQL `JOIN`)

> **大白话口诀**：
> 核心是找到 `on='牵线红娘的列名'`。`how=` 决定了如果这相亲配不成，是对不上的踢掉 (`inner`) 还是保留主表的人 (`left`)。

In [ ]:
# 1. 👈 [左连接 Left Join]：以主表(df_A)为主，从 df_B 把单价抄过来。（万能绝招）
# df_merged = pd.merge(df_A, df_B, on='硬件类型', how='left')

# 2. 🤝 [内连接 Inner Join]：只要双方都有的。比如找既在“暴利预警列表”里，又在我们数据库里的品牌。
# df_profitable = pd.merge(df_brands, df_thresholds, on='品牌', how='inner')

# 3. 🐍 [自连接 Self Join的黑魔法]：同款电脑，用它高核显版的价格 减去 它的低核显版的价格。
#    通过 `suffixes` 自动帮你改名叫 '价格_高配', '价格_标配'
"""
df_high = df[df['内存'] == 24]
df_low = df[df['内存'] == 16]
df_diff = pd.merge(df_high, df_low, on='机型', suffixes=('_高配', '_标配'))
df_diff['厂家白赚你的钱'] = df_diff['价格_高配'] - df_diff['价格_标配']
"""

---
## 4️⃣ 底层超能计算 (`NumPy`)

> **大白话口诀**：当 Pandas 处理复杂业务规则，或者要造数时，用 `np` 极速起飞。

---
## 🌟 补充篇：折叠分组与排序 (`groupby`)

> **大白话口诀**：这是纯正的 SQL `GROUP BY` 的镜像。

**SQL 视角：**
```sql
SELECT 季度, SUM(销量) AS 汇总销量 
FROM df 
GROUP BY 季度 
ORDER BY 汇总销量 ASC;
```

**Pandas 镜像翻译：**
* `GROUP BY 季度` ➡️ `groupby('季度')`
* `SELECT SUM(销量)` ➡️ `['销量'].sum()`
* `ORDER BY ... ASC` ➡️ `.sort_values(ascending=True)`

In [ ]:
# [Pandas 1行搞定 SQL 的 GROUP BY + ORDER BY]
# df.groupby('季度')['销量'].sum().sort_values(ascending=True)

In [ ]:
import numpy as np

# 1. 🚦 [最强打标神器1：两档开关]
# np.where(条件, 是的话填什么, 不是的话填什么)
# df['定位'] = np.where(df['价格'] > 8000, '高端', '普通')

# 2. 🚦 [最强打标神器2：多档开关] (SQL CASE WHEN)
"""
conditions = [
    (df['价格'] >= 10000), 
    (df['价格'] >= 6000) & (df['价格'] < 10000), 
    (df['价格'] < 6000)
]
choices = ['顶级尊享', '主流配置', '垃圾入门']
df['细分市场'] = np.select(conditions, choices, default='其他未知')
"""

---
## 5️⃣ SQL / Pandas 降维打击 (上帝视角的窗口函数/分组)

> **大白话口诀**：
> 不要把原表“折叠/弄短”，而是在最后面**加一列全局或者组内统计值**，好让当前行直接跟全局去比（比如：我比平均高多少？）。

In [ ]:
# =========【SQL 视角】=============

# 1. 📊 [求全市场占比]：
# ROUND(产值 * 100.0 / SUM(产值) OVER(), 2) AS 市场占比

# 2. 👥 [求自己在各自品牌的内部排名]：
# DENSE_RANK() OVER(PARTITION BY 品牌 ORDER BY 销量 DESC) AS 组内销冠排名

# 3. ❄️ [滚雪球/累计求和]：
# SUM(营业额) OVER(ORDER BY 营业额 DESC) AS 累计占领金额

# =========【Pandas 视角】=============

# 1. 🌍 [Pandas 版的 PARTITION BY]：用 `transform`，算出来长度和原表一模一样！
# df['本群均价'] = df.groupby('品牌')['价格'].transform('mean')
# df['品牌内溢价百分比'] = df['价格'] / df['本群均价']

---
## 6️⃣ 文本挖掘与特征化 (清洗脏数据/提炼关键词)

> **大白话口诀**：
> 不要用 `for` 循环和 `apply` 去一行行查字！用 Pandas 自带的 `.str` 文本方法，速度快 100 倍。这对应 SQL 里的 `LIKE '%关键词%'`。

In [ ]:
# 1. 🧹 [清理列名的空格 (第一步必做)]：
# df.columns = df.columns.str.strip()

# 2. 🔍 [Pandas版 LIKE '%提取%']：判断某个字串是否包含 "RTX 50" 或 "GDDR 7"
# 你的原版写法是用 apply + for 循环，数据量大时会很慢。
# 【进阶秒杀版】直接用 .str.contains 加管道符 | (代表"或")
"""
keywords = 'RTX 50|GDDR 7'
# 只要文本里包含其中一个关键词，就返回 True(1)，否则 False(0)
df['含有最新显卡'] = df['显卡名称'].str.upper().str.contains(keywords, na=False).astype(int)
"""

# =========【SQL 视角对照】=============
"""
SELECT *, 
    CASE 
        WHEN UPPER(显卡名称) LIKE '%RTX 50%' THEN 1
        WHEN UPPER(显卡名称) LIKE '%GDDR 7%' THEN 1
        ELSE 0
    END AS 含有最新显卡
FROM df;
"""

---
## 7️⃣ 行列转换：长宽表互换 (`pivot_table` & `melt`)

> **大白话口诀**：
> *   `pivot_table` (长变宽)：把**某列里的值**变成**表头**（做做报表/给人看专用）。
> *   `melt` (宽变长)：把**一排表头**融化成**一列里的值**（存进数据库/画大屏图表专用）。
>
> ⚠️ **避坑指南**：如果你自己用代码 `pivot` 出了宽表，之后又要去 `melt`，中间务必加一句 `.rename_axis(None, axis=1)` 抹掉隐形的列名标签！如果是直接 `pd.read_excel()` 读出来的干净宽表，直接 `melt` 即可！

In [ ]:
# 1. 📈 [长表变宽表 (做报表)]：透视表 `pivot_table`
# 把 '季度' 里的名字 (Q1、Q2) 拆出来挂到表头，当新列名。
# 【SQL 视角】 类似于复杂的 CASE WHEN sum。
'''
df_wide = df.pivot_table(
    index='品牌',       # 横向的行标 (相当于 GROUP BY 品牌)
    columns='季度',     # 纵向的列标 (谁的内容挂在表头上)
    values='销售额',    # 中间的格子填什么数字
    aggfunc='sum',      # 重复的怎么算 (求和、平均)
    margins=True,       # 自动生成 "All" (总计) 这一行/列
    fill_value=0        # 如果有空格子，填上0
)
# 🎉 打扫战场环节 (如果是给下面 melt 铺垫的话)：
df_wide = df_wide.reset_index()               # 把 '品牌' 从黑粗体索引变回普通列
df_wide = df_wide.rename_axis(None, axis=1)   # 把悬在表头上的 '季度' 标签擦掉！
'''

# 2. 📉 [宽表变长表 (存库)]：反透视融化 `melt`
# 把所有的 '季度' 比如 Q1, Q2 (除了'品牌'之外的所有列) 融化成两列。
'''
df_long = df_wide.melt(
    id_vars=['品牌'],       # 固定不动的列 (保留当参照物的)
    var_name='季度_名称',   # 融化下来的那些表头 (Q1,Q2)，新列名叫什么？
    value_name='当季销售额' # 融化下来的那些格子里的数字，新列名叫什么？
)
'''

---
### 💡 附：从纯 SQL 角度理解 Pandas 透视表的“降维打击”

如果不用 `pivot_table`，要实现“行转列”（按地区或季度），纯 SQL 只能写无数个 `CASE WHEN` 交叉表，维护堪称地狱模式。

更可怕的是**求平均的致命陷阱 (AVG Bug)**：
* **求 SUM 时：** `SUM(CASE WHEN 季度='Q1' THEN 销售额 ELSE 0 END)`没问题。
* **求 AVG 时：如果照抄加了 `ELSE 0`** ⬇️👇
  `AVG(CASE WHEN 季度='Q1' THEN 销售额 ELSE 0 END)` 
  👉 **大错特错！** 这会把非 Q1 的月份当做 `0` 并入计算，导致你的分母变大，平均值严重变小！必需写成 `ELSE NULL` 或直接省掉 ELSE。

但在 Pandas 里，这一切不需要写 CASE WHEN 狂魔版，只要：
`aggfunc=['sum', 'mean']` ———— 搞定！

---
## 8️⃣ 脏字符清理与强转数字 (巧用正则)

> **大白话口诀**：Excel 导出的数据经常带 `￥`、`%`、`万元` 或千分号 `,`。别手写循环！直接用 `.str.replace()` 配合正则 (Regex) 一指剥离，再挂上 `.astype()` 强行转成算术数字。

In [ ]:
# ⚠️ 场景：你拿到一列价格：["￥12,500.00", "￥998", "免费"]
# 目标：把前两个转成真实的数字12500.00，把"免费"变为空值或0。

# 1. 🧹 [多余字符一键替换（带正则版）]：
# 假设我们要剥离掉 "￥" 和 千分位逗号 "," 
# 注意：这里返回的不是布尔值！它是把字符替换掉后返回新的【纯净字符串】！
# ".str.replace" 加上 regex=True，允许使用正则表达式过滤
# "[\￥,]" 表示匹配到 "￥" 或者 "," 的地方，全部替换成空串 "" 
"""
df['干净的价格'] = df['乱七八糟的价格'].str.replace(r'[\￥,]', '', regex=True)
"""

# 2. 🔢 [强制转成数字并处理异常值]：
# 如果有的格子是 "暂无报价" 等纯文字，用 astype 转数字会报错。
# `pd.to_numeric(列, errors='coerce')` 可以强制转换！
# 遇到像"免费"或者"暂无报价"这种没法变数字的文字，自动帮你变成 NaN。
"""
df['价格_最终版'] = pd.to_numeric(df['干净的价格'], errors='coerce')
"""

# 3. 🎯 【一句话连招绝杀版】(最常写的方式)：
# 很多大牛直接把替换和转换连起来写：
"""
df['价格'] = pd.to_numeric(
    df['原始价格'].str.replace(r'[¥,]', '', regex=True), 
    errors='coerce'
)
"""

---
## 9️⃣ 时间提取与降维打击 (`.dt` 大法)

> **大白话口诀**：老板要按“月”看报表，但你的数据是 `"2026-03-23 15:30:00"` 这种精确到秒的怎么办？
> 不要用 `str` 切割！先用 `pd.to_datetime` 把它变成 Pandas 认识的时间老人，然后用 `.dt.year / .dt.month` 瞬间抽取出你想要的部分来做 `groupby`！

In [ ]:
# 1. 🧙‍♂️ [点石成金：字符串变时间格式]
# 第一步永远是把 "2026-03-23" 这种文本字符串，转化为被 Pandas 认可的时间核心类型 (datetime64)
# df['下单时间'] = pd.to_datetime(df['下单时间'])

# 2. ✂️ [切片大师：用 .dt 拿取任意时间颗粒度]
# 前提：你的列已经是 datetime 格式了，然后加上 .dt 就能像神仙一样随便拿
'''
df['年份'] = df['下单时间'].dt.year            # 提取结果: 2026
df['月份'] = df['下单时间'].dt.month           # 提取结果: 3
df['几号'] = df['下单时间'].dt.day             # 提取结果: 23
df['星期几'] = df['下单时间'].dt.dayofweek      # 提取结果: 0 (代表周一, 6代表周日)
df['只拿日期'] = df['下单时间'].dt.date          # 把时分秒砍掉，只剩 2026-03-23
'''

# 3. 📊 [实战连招]：老板要看每个月的总销售额怎么办？
'''
# 先造出一个"月份"列，然后直接用来做 Groupby 即可！
df['交易月'] = pd.to_datetime(df['购买时间']).dt.month
monthly_sales = df.groupby('交易月')['金额'].sum()
'''

---
## 🔟 进阶解惑：Pandas 的 "双框" 还是 "单框" (`[]` vs `[[]]`)

> **大白话口诀**：这是无数新手在 `merge` 和提取数据时疯狂报错的根源。
> 记住：**算数画单框 (变面条)，做表/合并画双框 (保尊严)！**

In [ ]:
# 1. 🍜 [画单框：抽出来的是一条"面条" (Series)]
# 什么时候用？—— 当你想拿这列数字去求和、求平均，或者做逻辑判断时。
"""
s = df['价格']           # 这是一根面条 (Series)
total = df['价格'].sum() # 把面条首尾相加
"""

# 2. 🍱 [画双框：抽出来的是一张"二维表" (DataFrame)]
# 什么时候用？—— 当你要把它塞进 `pd.merge()` 里去和别的表相亲，或者想保留表格的排版格式时！
# 因为 merge 强性规定：参与合并的双方都必须是 DataFrame(表)！
"""
df_table = df[['价格']]  # 此刻哪怕只有一列，它也带着表格边框的尊严 (DataFrame)
# 只有这样才能去和其他表做 JOIN：
result = pd.merge(df_A, df_B[['用户ID', '价格']], on='用户ID', how='inner')
"""

---
## 1️⃣1️⃣ SQL 心法：神级连招 CTE (`WITH`) 

> **大白话口诀**：告别俄罗斯套娃（嵌套子查询）。只要你想按步骤做数据清洗，就在 SQL 的开头用 `WITH` 建立备菜盘。
> 它本质上就等于你在 **Pandas 里新建好几个临时 DataFrame 变量 (`df1 = ...`, `df2 = ...`)** 然后拼在一起！

In [ ]:
# =========【SQL 视角：流水线写法】=============
'''sql
WITH 
-- 🔪 备菜盘 A（等同于 Pandas 里的 df1 = ...）
Total_Sales AS (
    SELECT 用户ID, SUM(金额) AS 消费总额
    FROM orders
    GROUP BY 用户ID
),

-- 🔪 备菜盘 B（基于上面的 A 继续操作，等同于 df2 = ...）
Top_VIPs AS (
    SELECT 用户ID 
    FROM Total_Sales
    ORDER BY 消费总额 DESC
    LIMIT 2
)

-- 🍳 热油下锅（等同于 Pandas 最后的 pd.merge 动作）
SELECT o.* 
FROM orders o 
INNER JOIN Top_VIPs v ON o.用户ID = v.用户ID;
'''

# =========【Pandas 镜像对应】=============
'''python
# 对应备菜盘 A
total_sales = df.groupby('用户ID')['金额'].sum().reset_index()

# 对应备菜盘 B
top_vips = total_sales.sort_values(by='金额', ascending=False).head(2)

# 对应最终下锅匹配 (注意这里用来匹配的右表加了双框！保尊严！)
final_result = pd.merge(df, top_vips[['用户ID']], on='用户ID', how='inner')
'''

### 📂 12. 自动化办公：批量处理/转化文件夹下的文件 (`os` + `for`)

> **大白话口诀**：先用 `os` 挨个点名列出文件名，再根据“后缀名”过滤，最后配合 Pandas 的 `read` 和 `to` 实现“格式大挪移”。

**核心逻辑：**
1. **定坐标**: `os.listdir(path)` 拿到全名单。
2. **设门禁**: `if filename.endswith(".xlsx")` 只拿 Excel。
3. **换马甲**: `os.path.splitext(filename)[0]` 提取纯文件名。
4. **防乱码**: `encoding='utf-8-sig'` 解决 Excel 打开 CSV 中文变成天书的问题。

In [ ]:
import os
import pandas as pd

# 💡 技巧：自动获取当前脚本/Notebook 所在的文件夹
# 在脚本中用: os.path.dirname(os.path.abspath(__file__))
target_dir = r"C:\你的数据文件夹路径"

def batch_convert_xlsx_to_csv(path):
    # 拿到所有 xlsx 文件名，并排除正在打开的临时文件 (~$)
    files = [f for f in os.listdir(path) if f.endswith(".xlsx") and not f.startswith("~$")]
    
    for filename in files:
        file_path = os.path.join(path, filename)
        
        # 📂 1. 读取 Excel
        df = pd.read_excel(file_path)
        
        # 🔗 2. 构造 CSV 路径 (把后缀换成 .csv)
        pure_name = os.path.splitext(filename)[0]
        csv_path = os.path.join(path, f"{pure_name}.csv")
        
        # ✍️ 3. 保存 CSV (编码选 utf-8-sig 才能在 Excel 里看中文)
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"✅ 已完成: {filename} -> {pure_name}.csv")

# batch_convert_xlsx_to_csv(target_dir) # 取消注释运行

### 🗄️ 13. 数据库高级：游标 (Cursor) 与 CTAS 建表

> **大白话口诀**：
> *   `Cursor` = 你派进数据库干活的“监工”，负责传递 SQL 指令。
> *   `CTAS` (Create Table As Select) = 把查出来的结果直接“固化”成一张新表，存在数据库里，而不是只留在 Python 内存里。

**核心逻辑：**
1. **建连接**: `psycopg2.connect(...)` 敲开数据库大门。
2. **派监工**: `conn.cursor()` 获取执行器。
3. **下指令**: `cur.execute(DDL/DML)` 执行建表/插数/删表。
4. **敲锤子**: `conn.commit()` 提交事务（不写这一行，数据库不认账！）。
5. **拖结果**: `pd.read_sql(...)` 把数据库里刚算好的成品表拉回 Pandas。

In [ ]:
import psycopg2
import pandas as pd

# 1. 🔑 [连大门]：建立数据库连接
conn = psycopg2.connect(
    host='localhost',
    port='5432',
    database='your_database',
    user='your_user',
    password='your_password'
)

# 2. 👷 [派监工]：创建游标对象
cur = conn.cursor()

# 3. 🔨 [CTAS 建表]：在数据库里直接算好并存成新表
# DROP + CREATE TABLE AS 是经典组合拳，先删旧再建新，保证可重复运行。
query_ctas = """
DROP TABLE IF EXISTS hot_drink_retail.report_loc_temp;

CREATE TABLE hot_drink_retail.report_loc_temp AS
SELECT 
    地区类型,
    -- 极寒挑战期：最低温 < -5度时的平均预售
    ROUND(AVG(CASE WHEN 最低温 < -5 THEN 预售量 END), 2) AS "1_极寒挑战期",
    
    -- 降温敏感期：-5度 ~ 5度
    ROUND(AVG(CASE WHEN 最低温 >= -5 AND 最低温 < 5 THEN 预售量 END), 2) AS "2_降温敏感期",
    
    -- 常温成长期：最低温 >= 5度
    ROUND(AVG(CASE WHEN 最低温 >= 5 THEN 预售量 END), 2) AS "3_常温成长期",

    -- 抗寒韧性 = 极寒期 / 常温期 * 100%（数值越高说明越不怕冷）
    ROUND(
        AVG(CASE WHEN 最低温 < -5 THEN 预售量 END) * 100.0 /
        NULLIF(AVG(CASE WHEN 最低温 >= 5 THEN 预售量 END), 0), 2
    ) AS "抗寒韧性_百分比"
FROM hot_drink_retail.source_data
GROUP BY 地区类型;
"""
cur.execute(query_ctas)

# 4. ✍️ [敲锤子]：提交事务（DDL 通常自动提交，但养成好习惯！）
conn.commit()

# 5. 🚚 [拖结果]：从数据库把刚建好的成品表拉回 Pandas
df_result = pd.read_sql("SELECT * FROM hot_drink_retail.report_loc_temp", conn)

# 6. 🧹 [打扫战场]：用完记得关门关灯
cur.close()
conn.close()

### 🕸️ 14. 全量沙盒：笛卡尔积 (Cross Join)

> **大白话口诀**：
> *   **Cross Join** = 把 A 表的每一行和 B 表的每一行**两两配对**，一个都不落下！
> *   结果行数 = `A的行数 × B的行数`。小心爆炸！通常用于造测试数据/枚举所有可能组合。

**场景举例：**
> 你有 3 个城市 × 4 个季度，想造一张 `3 × 4 = 12 行` 的“全量城市-季度”骨架表，后面再往里填数。

In [ ]:
import pandas as pd

# 准备两张小表
df_cities = pd.DataFrame({'城市': ['北京', '上海', '深圳']})
df_quarters = pd.DataFrame({'季度': ['Q1', 'Q2', 'Q3', 'Q4']})

# =====================================================
# 方法一：Pandas 专属写法 (推荐) — `how='cross'`
# =====================================================
df_cross = pd.merge(df_cities, df_quarters, how='cross')
print("--- Pandas Cross Join 结果 (3×4=12行) ---")
print(df_cross)

# =====================================================
# 方法二：万能土法 — 先造一个临时 key 列，按 left 连再删掉
# 适合老旧版本 Pandas (不支持 cross) 或者写 SQL
# =====================================================
df_cities['临时钥匙'] = 1
df_quarters['临时钥匙'] = 1
df_cross_2 = pd.merge(df_cities, df_quarters, on='临时钥匙', how='left').drop('临时钥匙', axis=1)

# =====================================================
# SQL 对照版
# =====================================================
'''sql
-- 现代 SQL (PostgreSQL / MySQL 8.0+)
SELECT * FROM cities CROSS JOIN quarters;

-- 老旧 SQL (通用土法)
SELECT a.*, b.*
FROM cities a, quarters b;
'''